# 01 – Overview: what does the data look like?

Before looking for patterns: verdict counts, train vs. test attribute distributions (should match, since the split is
random), and error rate per single attribute for both splits. Error rates come with 95% Wilson intervals; bins with
fewer than `MIN_N` examples are drawn hollow so tiny bins don't get over-read.

Needs `data/predictions_test.csv` and `data/predictions_train.csv` (`python src/make_predictions.py --split train`).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# find the repo root (works from phase1/ai-aided/ or anywhere below the repo)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "predictions_test.csv").exists())
FIG = Path.cwd() / "figures"
FIG.mkdir(exist_ok=True)

train = pd.read_csv(ROOT / "data" / "predictions_train.csv")
test = pd.read_csv(ROOT / "data" / "predictions_test.csv")
df = pd.concat([train, test], ignore_index=True)
df["error"] = ~df["correct"]

ATTRS = ["len", "n_vars", "n_terms", "max_depth", "max_const",
         "n_and", "n_or", "n_xor", "n_not", "n_add", "n_sub", "n_mul"]
SPLITS = {"train": "tab:blue", "test": "tab:orange"}
MIN_N = 30  # bins with fewer examples are drawn hollow

df.groupby("split").size()

## 1. Verdict counts

In [ ]:
ORDER = ["exact", "equivalent", "wrong", "invalid", "unknown"]
counts = pd.crosstab(df["verdict"], df["split"]).reindex(ORDER, fill_value=0)[list(SPLITS)]
share = counts / counts.sum()
display(pd.concat({"count": counts, "share": (100 * share).round(2)}, axis=1))
print("error rate:", {s: f"{df.loc[df.split == s, 'error'].mean():.2%}" for s in SPLITS})

fig, ax = plt.subplots(figsize=(7, 3))
y = np.arange(len(ORDER))
for i, (s, color) in enumerate(SPLITS.items()):
    vals = share[s].to_numpy()
    ax.barh(y + (i - 0.5) * 0.38, np.where(vals > 0, vals, np.nan), height=0.38, color=color, label=s)
    for yy, v, n in zip(y, vals, counts[s]):
        if n:
            ax.text(v * 1.15, yy + (i - 0.5) * 0.38, f"{n:,}", va="center", fontsize=8)
ax.set_xscale("log")  # 'exact' dominates; log scale keeps the rare verdicts visible
ax.set_yticks(y, ORDER)
ax.invert_yaxis()
ax.set_xlabel("share of split (log scale)")
ax.set_title("Verdicts by split")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIG / "verdicts_by_split.png", dpi=150)

## 2. Train vs. test distributions

One bin per integer value where possible. The total-variation distance (TVD, 0 = identical, 1 = disjoint) quantifies
"do they match?" – values around 0.01–0.02 are just sampling noise for a random split.

In [ ]:
def integer_edges(values, max_bins=25):
    lo, hi = int(values.min()), int(values.max())
    width = max(1, int(np.ceil((hi - lo + 1) / max_bins)))
    return np.arange(lo - 0.5, hi + width + 0.5, width)


def plot_distributions(prefix):
    fig, axes = plt.subplots(3, 4, figsize=(14, 8))
    tvd = {}
    for ax, attr in zip(axes.flat, ATTRS):
        col = f"{prefix}_{attr}"
        edges = integer_edges(df[col])
        dens = {}
        for s, color in SPLITS.items():
            v = df.loc[df.split == s, col]
            dens[s], _ = np.histogram(v, bins=edges)
            dens[s] = dens[s] / len(v)
            ax.stairs(dens[s], edges, color=color, lw=1.5, label=s)
        tvd[col] = 0.5 * np.abs(dens["train"] - dens["test"]).sum()
        ax.set_title(f"{col}  (TVD {tvd[col]:.3f})", fontsize=9)
        ax.set_ylabel("share", fontsize=8)
    axes.flat[0].legend()
    fig.suptitle(f"{prefix}: train vs. test distributions")
    fig.tight_layout()
    fig.savefig(FIG / f"distributions_{prefix}.png", dpi=150)
    return pd.Series(tvd, name="TVD")


tvd_src = plot_distributions("src")

In [ ]:
tvd_trg = plot_distributions("trg")
pd.concat([tvd_src, tvd_trg]).sort_values(ascending=False).round(3).to_frame().T

## 3. Error rate per attribute (train and test)

Line = error rate with 95% Wilson interval; hollow markers = fewer than `MIN_N` examples in that bin.
Faint lines on the right axis = number of examples per bin (log scale). Train errors are expected to be lower
(the model saw those examples) – where the gap between the two lines is large is itself interesting.

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return centre - half, centre + half


def error_table(col, split=None, max_bins=15):
    """Error rate per bin of `col` (optionally for one split), with counts and Wilson interval."""
    d = df if split is None else df[df.split == split]
    edges = integer_edges(df[col], max_bins)
    b = pd.cut(d[col], edges)
    t = d.groupby(b, observed=True)["error"].agg(n="size", errors="sum")
    t["error_rate"] = t["errors"] / t["n"]
    t["ci_low"], t["ci_high"] = wilson(t["errors"], t["n"])
    t["x"] = [iv.mid for iv in t.index]
    return t


def plot_error_rates(prefix):
    fig, axes = plt.subplots(3, 4, figsize=(15, 9))
    for ax, attr in zip(axes.flat, ATTRS):
        col = f"{prefix}_{attr}"
        ax_n = ax.twinx()
        for s, color in SPLITS.items():
            t = error_table(col, s)
            ax.plot(t.x, t.error_rate, color=color, lw=1, alpha=0.7)
            ax.vlines(t.x, t.ci_low, t.ci_high, color=color, alpha=0.4)
            big = t.n >= MIN_N
            ax.scatter(t.x[big], t.error_rate[big], color=color, s=18, zorder=3, label=s)
            ax.scatter(t.x[~big], t.error_rate[~big], facecolor="white", edgecolor=color, s=18, zorder=3)
            ax_n.plot(t.x, t.n, color=color, lw=0.8, alpha=0.25, drawstyle="steps-mid")
        ax_n.set_yscale("log")
        ax_n.tick_params(labelsize=7, colors="gray")
        ax.set_ylim(bottom=0)
        ax.set_title(col, fontsize=9)
        ax.set_ylabel("error rate", fontsize=8)
    axes.flat[0].legend(fontsize=8)
    fig.suptitle(f"Error rate per {prefix} attribute (95% Wilson CI; hollow = n < {MIN_N}; faint = n, log)")
    fig.tight_layout()
    fig.savefig(FIG / f"error_rate_{prefix}.png", dpi=150)


plot_error_rates("src")

In [ ]:
plot_error_rates("trg")

## 4. Inspect any single attribute as a table

In [ ]:
error_table("src_n_terms", "test").round(3)

## 5. Presentation versions (simpler plots for slides)

Test split only, one attribute per plot, bars instead of lines. Each bar shows the share of wrong outputs, its 95%
interval (black whisker) and the number of examples. Bins with fewer than `MIN_N` examples are left out
(the footnote says how many). The dashed line is the overall test error rate.

In [ ]:
def value_bins(s, width=1):
    """Bin integer values into blocks of `width`; returns (bin start, readable label like '3' or '20-24')."""
    start = (s // width) * width
    label = start.astype(str) if width == 1 else start.astype(str) + "-" + (start + width - 1).astype(str)
    return start.rename("start"), label.rename("label")


def clean_error_table(col, split="test", width=1, min_n=MIN_N):
    """Like error_table, but rows are labelled with plain values ('3', '20-24') instead of pandas intervals."""
    d = df[df.split == split]
    start, label = value_bins(d[col], width)
    t = d.groupby([start, label])["error"].agg(n="size", errors="sum")
    t.index = pd.Index(t.index.get_level_values("label"), name=col)
    t["error_rate"] = t["errors"] / t["n"]
    t["ci_low"], t["ci_high"] = wilson(t["errors"], t["n"])
    t["reliable"] = t["n"] >= min_n
    return t


def simple_error_plot(col, xlabel, width=1, split="test", min_n=MIN_N):
    t = clean_error_table(col, split, width, min_n)
    dropped = t.loc[~t.reliable, "n"].sum()
    t = t[t.reliable]
    x = np.arange(len(t))
    fig, ax = plt.subplots(figsize=(max(4.5, 0.6 * len(t) + 2), 3.6))
    ax.bar(x, t.error_rate, color=SPLITS[split], alpha=0.85, width=0.7)
    ax.errorbar(x, t.error_rate, yerr=[t.error_rate - t.ci_low, t.ci_high - t.error_rate],
                fmt="none", ecolor="black", capsize=3, lw=1)
    for xi, (r, n, hi) in enumerate(zip(t.error_rate, t.n, t.ci_high)):
        ax.text(xi, hi + 0.01, f"{r:.0%}\nn={n:,}", ha="center", va="bottom", fontsize=8)
    base = df.loc[df.split == split, "error"].mean()
    ax.axhline(base, ls="--", lw=1, color="gray")
    ax.text(len(t) - 0.5, base, f" overall {base:.1%}", va="bottom", ha="right", fontsize=8, color="gray")
    ax.set_xticks(x, t.index)
    ax.set_ylim(0, min(1.05, t.ci_high.max() + 0.15))
    ax.set_xlabel(xlabel)
    ax.set_ylabel(f"share of wrong outputs ({split})")
    ax.spines[["top", "right"]].set_visible(False)
    if dropped:
        fig.text(0.01, 0.01, f"{dropped} examples in bins with n < {min_n} not shown", fontsize=7, color="gray")
    fig.tight_layout()
    fig.savefig(FIG / f"pres_{col}.png", dpi=200)
    return t

In [ ]:
# A: overall error rate, train vs. test
summary = df.groupby("split")["error"].agg(n="size", errors="sum").reindex(list(SPLITS))
summary["error_rate"] = summary["errors"] / summary["n"]
fig, ax = plt.subplots(figsize=(4, 3.4))
ax.bar(summary.index, summary.error_rate, color=list(SPLITS.values()), width=0.6)
for i, (r, k, n) in enumerate(zip(summary.error_rate, summary.errors, summary.n)):
    ax.text(i, r + 0.003, f"{r:.1%}\n({k:,} of {n:,})", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, summary.error_rate.max() * 1.4)
ax.set_ylabel("share of wrong outputs")
ax.set_title("Seen (train) vs. unseen (test) examples")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIG / "pres_overall.png", dpi=200)

In [ ]:
# B: number of variables in the input
simple_error_plot("src_n_vars", "number of variables in the input")

In [ ]:
# C: input length (binned in steps of 5 characters)
simple_error_plot("src_len", "input length (characters)", width=5)

In [ ]:
# the same numbers as a table, with readable labels
clean_error_table("src_n_terms").round(3)